# LiteFallNet sin la capa GRU: entrenamiento con SisFall

Este cuaderno entrena la versión del artículo sin la capa GRU (unidad recurrente con compuertas).
`AVM` resume la aceleración de los tres ejes; `GVM`, el giro medido por el giroscopio.
`ADL` significa actividades de la vida diaria; `Fall`, caída.
Usa SisFall para entrenar y UPFall para una evaluación independiente.

Al final guarda el modelo, sus datos de preparación y las matrices necesarias
para el siguiente cuaderno.

Artículo de referencia:
Owusu, E., Acquah, I., Asare, M. A., & Yeboah, B. A. (2025). LiteFallNet: A lightweight deep learning model for efficient real-time fall detection. Digital health, 11, 20552076251386698. https://doi.org/10.1177/20552076251386698

## 1. Librerías y ajustes

Las señales ya están a 50 Hz (50 muestras por segundo). Cada ventana tiene 150 muestras y avanza 75 muestras.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(42)

FRECUENCIA_HZ = 50
TAMANO_VENTANA = 150
PASO_VENTANA = 75
MARGEN_CAIDA_SEGUNDOS = 1.5
MARGEN_CAIDA_MUESTRAS = int(MARGEN_CAIDA_SEGUNDOS * FRECUENCIA_HZ)
CANALES = ["AVM", "GVM"]
COLUMNAS_TRIAL = ["Dataset", "Subject", "Activity_Code", "Trial"]
COLUMNAS_REQUERIDAS = COLUMNAS_TRIAL + ["Activity_Label", "Sample_Index"] + CANALES

# Cada sensor tiene tres ejes; la norma máxima es la raíz de tres veces el límite por eje.
ACELEROMETRO_LIMITE_EJE_G = 16.0
GIROSCOPIO_LIMITE_EJE_GRADOS_SEGUNDO = 2000.0
NORMA_ACELEROMETRO = float(np.sqrt(3) * ACELEROMETRO_LIMITE_EJE_G)
NORMA_GIROSCOPIO = float(np.sqrt(3) * GIROSCOPIO_LIMITE_EJE_GRADOS_SEGUNDO)

SEMILLA = 42
EPOCAS_MAXIMAS = 300
TAMANO_LOTE = 32
TASA_APRENDIZAJE = 1e-4
UMBRAL_CAIDA = 0.5

# Buscamos la raíz del proyecto para que las rutas funcionen desde ambas carpetas.
RAIZ_PROYECTO = Path.cwd()
while not (RAIZ_PROYECTO / "pyproject.toml").is_file() and RAIZ_PROYECTO.parent != RAIZ_PROYECTO:
    RAIZ_PROYECTO = RAIZ_PROYECTO.parent
CARPETA_NOTEBOOKS = RAIZ_PROYECTO / "notebooks"
RUTA_SET_A = CARPETA_NOTEBOOKS / "data" / "oro" / "falls" / "set_a.parquet"
RUTA_SALIDA = CARPETA_NOTEBOOKS / "data" / "modelos" / "falls" / "litefallnet_sisfall"

print(f"Ventana: {TAMANO_VENTANA} muestras; paso: {PASO_VENTANA} muestras")
print(f"Norma AVM: {NORMA_ACELEROMETRO:.3f} g; norma GVM: {NORMA_GIROSCOPIO:.3f} grados/s")
print(f"Archivo de entrada: {RUTA_SET_A}")

## 2. Funciones de preparación y modelo

Primero se separan ensayos completos. Después se crean las ventanas para cada grupo.

In [ ]:
def validar_dataset(df):
    """Comprueba las columnas y etiquetas necesarias para este cuaderno."""
    faltantes = set(COLUMNAS_REQUERIDAS) - set(df.columns)
    if faltantes:
        raise ValueError(f"Faltan columnas necesarias: {sorted(faltantes)}")
    if df[COLUMNAS_REQUERIDAS].isna().any().any():
        raise ValueError("Las columnas necesarias no pueden tener valores vacíos.")
    if not np.isfinite(df[CANALES].to_numpy(dtype=np.float64)).all():
        raise ValueError("AVM y GVM contienen valores no finitos (NaN o infinito).")
    etiquetas = set(df["Activity_Label"].unique())
    if not etiquetas.issubset({"ADL", "Fall"}):
        raise ValueError(f"Hay etiquetas no reconocidas: {sorted(etiquetas)}")


def escalar_senales(df):
    """Divide cada magnitud por la norma máxima de su sensor."""
    resultado = df.copy()
    resultado["AVM"] = pd.to_numeric(resultado["AVM"], errors="raise").astype(np.float32) / NORMA_ACELEROMETRO
    resultado["GVM"] = pd.to_numeric(resultado["GVM"], errors="raise").astype(np.float32) / NORMA_GIROSCOPIO
    return resultado


def dividir_por_trials(df):
    """Asigna cada ensayo completo a entrenamiento, validación o prueba."""
    etiquetas_por_trial = df.groupby(COLUMNAS_TRIAL, sort=False)["Activity_Label"].nunique()
    if (etiquetas_por_trial != 1).any():
        raise ValueError("Cada trial debe tener una sola etiqueta.")

    trials = df[COLUMNAS_TRIAL + ["Activity_Label"]].drop_duplicates().reset_index(drop=True)
    etiquetas = trials["Activity_Label"].to_numpy()
    conteo = pd.Series(etiquetas).value_counts()
    if len(conteo) != 2 or conteo.min() < 5:
        raise ValueError("Se necesitan al menos cinco trials de cada clase para dividir 60/20/20.")

    indices = np.arange(len(trials))
    indices_train, indices_resto = train_test_split(
        indices, test_size=0.4, random_state=SEMILLA, stratify=etiquetas
    )
    indices_validacion, indices_prueba = train_test_split(
        indices_resto,
        test_size=0.5,
        random_state=SEMILLA,
        stratify=etiquetas[indices_resto],
    )

    particiones = np.empty(len(trials), dtype=object)
    particiones[indices_train] = "train"
    particiones[indices_validacion] = "validation"
    particiones[indices_prueba] = "test"
    trials["particion"] = particiones
    return df.merge(trials[COLUMNAS_TRIAL + ["particion"]], on=COLUMNAS_TRIAL, how="left", validate="many_to_one")


def validar_indices_trial(trial):
    """Rechaza índices repetidos o saltos dentro de un ensayo."""
    indices = trial["Sample_Index"].to_numpy(dtype=np.float64)
    if not np.isfinite(indices).all() or not np.equal(indices, np.floor(indices)).all():
        raise ValueError("Sample_Index debe contener números enteros.")
    if pd.Index(indices).has_duplicates:
        raise ValueError("Hay Sample_Index duplicados dentro de un ensayo.")
    if len(indices) > 1 and not np.all(np.diff(indices) == 1):
        raise ValueError("Los Sample_Index de cada ensayo deben ser consecutivos.")


def crear_ventanas(df):
    """Crea ventanas sin cruzar ensayos y marca caídas cerca del pico AVM."""
    ventanas = []
    etiquetas = []

    for _, trial in df.groupby(COLUMNAS_TRIAL, sort=False):
        trial = trial.copy()
        trial["Sample_Index"] = pd.to_numeric(trial["Sample_Index"], errors="raise")
        trial = trial.sort_values("Sample_Index")
        validar_indices_trial(trial)
        senales = trial[CANALES].to_numpy(dtype=np.float32)
        inicios = np.arange(0, len(trial) - TAMANO_VENTANA + 1, PASO_VENTANA)
        if len(inicios) == 0:
            continue

        indices = inicios[:, None] + np.arange(TAMANO_VENTANA)[None, :]
        ventanas.append(senales[indices])

        if trial["Activity_Label"].iloc[0] == "Fall":
            pico = int(np.argmax(trial["AVM"].to_numpy(dtype=np.float32)))
            centros = inicios + TAMANO_VENTANA // 2
            etiquetas.append((np.abs(centros - pico) <= MARGEN_CAIDA_MUESTRAS).astype(np.int8))
        else:
            etiquetas.append(np.zeros(len(inicios), dtype=np.int8))

    if not ventanas:
        return (
            np.empty((0, TAMANO_VENTANA, len(CANALES)), dtype=np.float32),
            np.empty((0,), dtype=np.int8),
        )
    return np.concatenate(ventanas).astype(np.float32), np.concatenate(etiquetas).astype(np.int8)


def mostrar_conteos(nombre, y):
    """Muestra cuántas ventanas hay de cada clase."""
    conteo = np.bincount(np.asarray(y, dtype=np.int8), minlength=2)
    print(f"{nombre}: ADL={conteo[0]:,}; caídas={conteo[1]:,}")


def agregar_ruido_a_caidas(x, y, copias=2, sigma=0.01):
    """Añade copias con ruido gaussiano solo a las ventanas de caída."""
    rng = np.random.default_rng(SEMILLA)
    ventanas_caida = x[y == 1]
    if len(ventanas_caida) == 0:
        raise ValueError("No hay ventanas de caída para aumentar.")

    copias_ruidosas = [
        ventanas_caida + rng.normal(0, sigma, size=ventanas_caida.shape).astype(np.float32)
        for _ in range(copias)
    ]
    x_aumentado = np.concatenate([x, *copias_ruidosas]).astype(np.float32)
    y_aumentado = np.concatenate([y, np.ones(copias * len(ventanas_caida), dtype=np.int8)])
    orden = rng.permutation(len(y_aumentado))
    return x_aumentado[orden], y_aumentado[orden]


def seleccionar_calibracion(x, y, caidas=600, adl=400):
    """Toma 600 caídas y 400 ADL del entrenamiento original."""
    rng = np.random.default_rng(SEMILLA)
    indices_caidas = np.flatnonzero(y == 1)
    indices_adl = np.flatnonzero(y == 0)
    if len(indices_caidas) < caidas or len(indices_adl) < adl:
        raise ValueError("Se necesitan 600 ventanas de caída y 400 ventanas ADL para calibrar.")
    elegidos = np.concatenate(
        [
            rng.choice(indices_caidas, size=caidas, replace=False),
            rng.choice(indices_adl, size=adl, replace=False),
        ]
    )
    rng.shuffle(elegidos)
    return x[elegidos].astype(np.float32), y[elegidos].astype(np.int8)


def construir_modelo():
    """Construye la versión del artículo sin la capa GRU."""
    entradas = layers.Input(shape=(TAMANO_VENTANA, len(CANALES)), name="entrada")
    x = layers.BatchNormalization(name="normalizacion_entrada")(entradas)

    residual = layers.Conv1D(32, 1, padding="same", name="proyeccion_residual")(x)
    x = layers.Conv1D(32, 3, dilation_rate=2, padding="causal", name="convolucion_causal_1")(x)
    x = layers.BatchNormalization(name="normalizacion_1")(x)
    x = layers.LeakyReLU(negative_slope=0.2, name="leaky_relu_causal_1")(x)
    x = layers.Conv1D(32, 3, dilation_rate=2, padding="causal", name="convolucion_causal_2")(x)
    x = layers.BatchNormalization(name="normalizacion_2")(x)
    x = layers.Add(name="suma_residual")([x, residual])
    x = layers.LeakyReLU(negative_slope=0.2, name="leaky_relu_causal_2")(x)

    atencion = layers.GlobalAveragePooling1D(name="promedio_para_se")(x)
    atencion = layers.Dense(32 // 16, activation="relu", name="se_reduccion")(atencion)
    atencion = layers.Dense(32, activation="sigmoid", name="se_ampliacion")(atencion)
    atencion = layers.Reshape((1, 32), name="se_forma")(atencion)
    x = layers.Multiply(name="se_ponderacion")([x, atencion])

    x = layers.SeparableConv1D(64, 3, padding="same", name="convolucion_separable")(x)
    x = layers.LeakyReLU(negative_slope=0.2, name="leaky_relu_separable")(x)
    x = layers.MaxPooling1D(pool_size=2, name="max_pooling")(x)
    x = layers.BatchNormalization(name="normalizacion_final")(x)
    maximo = layers.GlobalMaxPooling1D(name="maximo_global")(x)
    promedio = layers.GlobalAveragePooling1D(name="promedio_global")(x)
    x = layers.Concatenate(name="unir_pooling")([maximo, promedio])
    x = layers.Dense(64, name="densa_64")(x)
    x = layers.LeakyReLU(negative_slope=0.2, name="leaky_relu_densa")(x)
    salidas = layers.Dense(1, activation="sigmoid", name="probabilidad_caida")(x)

    modelo = tf.keras.Model(entradas, salidas, name="LiteFallNet_Ablacion_Sin_GRU_Articulo")
    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=TASA_APRENDIZAJE),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return modelo


def calcular_metricas(y, probabilidades):
    """Calcula métricas binarias con caída como clase positiva."""
    y = np.asarray(y, dtype=np.int8)
    probabilidades = np.asarray(probabilidades, dtype=np.float32).ravel()
    predicciones = (probabilidades >= UMBRAL_CAIDA).astype(np.int8)
    matriz = confusion_matrix(y, predicciones, labels=[0, 1])
    verdaderos_negativos, falsos_positivos, _, _ = matriz.ravel()
    return {
        "accuracy": float(accuracy_score(y, predicciones)),
        "precision": float(precision_score(y, predicciones, pos_label=1, zero_division=0)),
        "recall": float(recall_score(y, predicciones, pos_label=1, zero_division=0)),
        "specificity": float(verdaderos_negativos / (verdaderos_negativos + falsos_positivos))
        if verdaderos_negativos + falsos_positivos
        else 0.0,
        "f1": float(f1_score(y, predicciones, pos_label=1, zero_division=0)),
        "auc": float(roc_auc_score(y, probabilidades))
        if len(np.unique(y)) == 2
        else float("nan"),
        "confusion_matrix": matriz,
    }


def evaluar_modelo(modelo, nombre, x, y):
    """Imprime métricas y muestra una matriz de confusión 2 por 2."""
    probabilidades = modelo.predict(x, batch_size=TAMANO_LOTE, verbose=0).ravel()
    metricas = calcular_metricas(y, probabilidades)
    print(f"{nombre}: umbral fijo = {UMBRAL_CAIDA}")
    print("Exactitud: {accuracy:.4f}; precisión: {precision:.4f}; recall de caídas: {recall:.4f}".format(**metricas))
    print("Especificidad: {specificity:.4f}; F1: {f1:.4f}; AUC: {auc:.4f}".format(**metricas))
    matriz = metricas["confusion_matrix"]
    print("Filas: etiqueta real; columnas: predicción; orden: ADL, caída.")
    print(matriz)
    pantalla = ConfusionMatrixDisplay(
        confusion_matrix=matriz, display_labels=["ADL", "Caída"]
    )
    pantalla.plot(cmap="Blues", values_format="d")
    plt.title(nombre)
    plt.tight_layout()
    plt.show()
    return metricas


## 3. Comprobación breve con datos sintéticos

Esta prueba no lee archivos ni entrena. Comprueba el tamaño de las ventanas, la
separación de trials, la escala de las señales y la forma de entrada del modelo.

In [ ]:
filas_sinteticas = []
for etiqueta in ("ADL", "Fall"):
    for numero_trial in range(5):
        cantidad = 450
        avm = np.full(cantidad, 2.0, dtype=np.float32)
        if etiqueta == "Fall":
            avm[225] = 15.0
        filas_sinteticas.append(
            pd.DataFrame(
                {
                    "Dataset": "SisFall",
                    "Subject": f"{etiqueta}-{numero_trial}",
                    "Activity_Code": "D01" if etiqueta == "ADL" else "F01",
                    "Trial": 1,
                    "Activity_Label": etiqueta,
                    "Sample_Index": np.arange(cantidad),
                    "AVM": avm,
                    "GVM": np.full(cantidad, 100.0, dtype=np.float32),
                }
            )
        )
df_sintetico = pd.concat(filas_sinteticas, ignore_index=True)
validar_dataset(df_sintetico)
try:
    validar_dataset(df_sintetico.assign(GVM=np.inf))
except ValueError as error:
    assert "infinito" in str(error)
else:
    raise AssertionError("La validación debe rechazar señales infinitas.")
df_sintetico = escalar_senales(df_sintetico)
df_particionado_sintetico = dividir_por_trials(df_sintetico)

ids_por_particion = {
    nombre: set(
        map(
            tuple,
            grupo[COLUMNAS_TRIAL].drop_duplicates().itertuples(index=False, name=None),
        )
    )
    for nombre, grupo in df_particionado_sintetico.groupby("particion")
}
assert ids_por_particion["train"].isdisjoint(ids_por_particion["validation"])
assert ids_por_particion["train"].isdisjoint(ids_por_particion["test"])
assert ids_por_particion["validation"].isdisjoint(ids_por_particion["test"])
conteos_trials = {
    nombre: grupo[COLUMNAS_TRIAL].drop_duplicates().shape[0]
    for nombre, grupo in df_particionado_sintetico.groupby("particion")
}
assert conteos_trials == {"train": 6, "validation": 2, "test": 2}
assert all(
    set(grupo["Activity_Label"].unique()) == {"ADL", "Fall"}
    for _, grupo in df_particionado_sintetico.groupby("particion")
)

x_sintetico, y_sintetico = crear_ventanas(df_sintetico)
assert x_sintetico.shape == (50, TAMANO_VENTANA, len(CANALES))
assert x_sintetico.dtype == np.float32
assert int(y_sintetico.sum()) == 15
assert np.isclose(
    df_sintetico.loc[df_sintetico["Subject"] == "Fall-0", "AVM"].max(),
    15.0 / NORMA_ACELEROMETRO,
)
assert np.isclose(
    df_sintetico.loc[df_sintetico["Subject"] == "ADL-0", "GVM"].iloc[0],
    100.0 / NORMA_GIROSCOPIO,
)
x_aumentado, y_aumentado = agregar_ruido_a_caidas(x_sintetico, y_sintetico)
assert int((y_aumentado == 0).sum()) == int((y_sintetico == 0).sum())
assert int((y_aumentado == 1).sum()) == 3 * int((y_sintetico == 1).sum())
modelo_prueba = construir_modelo()
assert modelo_prueba.name == "LiteFallNet_Ablacion_Sin_GRU_Articulo"
assert modelo_prueba.count_params() == 14_635
leaky_relu = [
    capa for capa in modelo_prueba.layers if isinstance(capa, layers.LeakyReLU)
]
assert len(leaky_relu) == 4
assert all(capa.negative_slope == 0.2 for capa in leaky_relu)
salida_sintetica = modelo_prueba(
    np.zeros((2, TAMANO_VENTANA, len(CANALES)), dtype=np.float32), training=False
)
assert tuple(salida_sintetica.shape) == (2, 1)
x_base_calibracion = np.zeros((1000, TAMANO_VENTANA, len(CANALES)), dtype=np.float32)
y_base_calibracion = np.concatenate(
    [np.ones(600, dtype=np.int8), np.zeros(400, dtype=np.int8)]
)
x_calibration_sintetico, y_calibration_sintetico = seleccionar_calibracion(
    x_base_calibracion, y_base_calibracion
)
assert x_calibration_sintetico.shape == (1000, TAMANO_VENTANA, len(CANALES))
assert np.array_equal(np.bincount(y_calibration_sintetico, minlength=2), [400, 600])
metricas_sinteticas = calcular_metricas(
    np.array([0, 0, 1, 1]), np.array([0.1, 0.8, 0.3, 0.9])
)
assert set(metricas_sinteticas) >= {
    "accuracy", "precision", "recall", "specificity", "f1", "auc"
}
print(
    f"Prueba correcta: ventanas={x_sintetico.shape}; "
    f"caídas antes/después={int((y_sintetico == 1).sum())}/{int((y_aumentado == 1).sum())}; "
    f"parámetros={modelo_prueba.count_params()}; calibración={len(y_calibration_sintetico)}; "
    f"salida={tuple(salida_sintetica.shape)}"
)

## 4. Leer y validar `set_a.parquet`

SisFall se reserva para entrenamiento y evaluación de prueba. UPFall solo se usa
para la evaluación independiente.

In [ ]:
df_set_a = pd.read_parquet(RUTA_SET_A)
validar_dataset(df_set_a)
df_sisfall = df_set_a.loc[df_set_a["Dataset"] == "SisFall", COLUMNAS_REQUERIDAS].copy()
df_upfall = df_set_a.loc[df_set_a["Dataset"] == "UPFall", COLUMNAS_REQUERIDAS].copy()
if df_sisfall.empty:
    raise ValueError("No hay filas de SisFall en set_a.parquet.")
if df_upfall.empty:
    raise ValueError("No hay filas de UPFall en set_a.parquet.")
df_sisfall = escalar_senales(df_sisfall)
df_upfall = escalar_senales(df_upfall)
print(f"SisFall: {len(df_sisfall):,} filas; UPFall: {len(df_upfall):,} filas")

## 5. Separar ensayos y crear ventanas

La división 60/20/20 ocurre antes del ventaneo. Así, las ventanas solapadas de
un mismo ensayo nunca aparecen en grupos distintos.

In [ ]:
df_sisfall_particionado = dividir_por_trials(df_sisfall)
for nombre in ("train", "validation", "test"):
    grupo = df_sisfall_particionado.loc[df_sisfall_particionado["particion"] == nombre]
    cantidad_trials = grupo[COLUMNAS_TRIAL].drop_duplicates().shape[0]
    print(f"{nombre}: {cantidad_trials:,} ensayos")

x_train_original, y_train_original = crear_ventanas(
    df_sisfall_particionado.loc[df_sisfall_particionado["particion"] == "train"]
)
x_validation, y_validation = crear_ventanas(
    df_sisfall_particionado.loc[df_sisfall_particionado["particion"] == "validation"]
)
x_test, y_test = crear_ventanas(
    df_sisfall_particionado.loc[df_sisfall_particionado["particion"] == "test"]
)
x_upfall, y_upfall = crear_ventanas(df_upfall)

if min(len(y_train_original), len(y_validation), len(y_test), len(y_upfall)) == 0:
    raise ValueError("Alguna partición no produjo ventanas. Revisa la duración de los trials.")
mostrar_conteos("Entrenamiento antes del aumento", y_train_original)
mostrar_conteos("Validación", y_validation)
mostrar_conteos("Prueba SisFall", y_test)
mostrar_conteos("Evaluación UPFall", y_upfall)

In [ ]:
x_train, y_train = agregar_ruido_a_caidas(x_train_original, y_train_original, copias=2, sigma=0.01)
mostrar_conteos("Entrenamiento después del aumento", y_train)

## 6. Entrenar LiteFallNet sin la capa GRU

El entrenamiento puede durar bastante. Se detiene pronto si la pérdida de validación
deja de mejorar después de la época 30.

In [ ]:
modelo = construir_modelo()
parada_temprana = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=20,
    start_from_epoch=30,
    restore_best_weights=True,
    verbose=1,
)
historial = modelo.fit(
    x_train,
    y_train,
    validation_data=(x_validation, y_validation),
    epochs=EPOCAS_MAXIMAS,
    batch_size=TAMANO_LOTE,
    callbacks=[parada_temprana],
    verbose=1,
)

## 7. Curva de aprendizaje y evaluación

El umbral se mantiene fijo en 0,5 para SisFall y UPFall. F1 resume el equilibrio entre precisión y detección de caídas; AUC (área bajo la curva) indica qué tan bien se separan las dos clases.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(historial.history["loss"], label="Entrenamiento")
plt.plot(historial.history["val_loss"], label="Validación")
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.title("Pérdida durante el entrenamiento")
plt.legend()
plt.tight_layout()
plt.show()

metricas_sisfall = evaluar_modelo(modelo, "Prueba SisFall", x_test, y_test)
metricas_upfall = evaluar_modelo(modelo, "Evaluación UPFall", x_upfall, y_upfall)

## 8. Nota para guardar el modelo con menos bits

El siguiente cuaderno guarda el modelo con enteros de 8 bits (INT8), que ocupan menos espacio. Si una métrica cae más de 2 puntos porcentuales, el modelo no está listo. Haría falta un ajuste especial durante el entrenamiento, llamado QAT (entrenamiento adaptado a la cuantización).

## 9. Guardar los artefactos

Los nombres de archivo y las claves del archivo comprimido se mantienen fijos para
que el siguiente cuaderno pueda leerlos.

In [ ]:
RUTA_SALIDA.mkdir(parents=True, exist_ok=True)
RUTA_MODELO = RUTA_SALIDA / "litefallnet_sisfall.keras"
RUTA_METADATA = RUTA_SALIDA / "litefallnet_sisfall_metadata.json"
RUTA_EVALUACION = RUTA_SALIDA / "litefallnet_sisfall_eval.npz"

modelo.save(RUTA_MODELO)

x_calibration, y_calibration = seleccionar_calibracion(
    x_train_original, y_train_original
)

np.savez_compressed(
    RUTA_EVALUACION,
    x_calibration=x_calibration,
    y_calibration=y_calibration,
    x_validation=x_validation.astype(np.float32),
    y_validation=y_validation.astype(np.int8),
    x_test=x_test.astype(np.float32),
    y_test=y_test.astype(np.int8),
    x_upfall=x_upfall.astype(np.float32),
    y_upfall=y_upfall.astype(np.int8),
)

metadata = {
    "modelo": "LiteFallNet: versión del artículo sin la capa GRU",
    "dataset_entrenamiento": "SisFall",
    "dataset_evaluacion_externa": "UPFall",
    "archivo_entrada": "set_a.parquet",
    "canales": CANALES,
    "forma_entrada": [TAMANO_VENTANA, len(CANALES)],
    "etiquetas": {"0": "ADL", "1": "Fall"},
    "normalizacion": {
        "AVM_límite_por_eje_g": ACELEROMETRO_LIMITE_EJE_G,
        "GVM_límite_por_eje_grados_segundo": GIROSCOPIO_LIMITE_EJE_GRADOS_SEGUNDO,
        "factor_norma_vectorial": float(np.sqrt(3)),
        "AVM_divisor": NORMA_ACELEROMETRO,
        "GVM_divisor": NORMA_GIROSCOPIO,
    },
    "ventana": {
        "frecuencia_hz": FRECUENCIA_HZ,
        "muestras": TAMANO_VENTANA,
        "paso_muestras": PASO_VENTANA,
        "margen_caida_segundos": MARGEN_CAIDA_SEGUNDOS,
        "margen_caida_muestras": MARGEN_CAIDA_MUESTRAS,
        "regla_etiqueta": "centro de ventana a no más de 1,5 s del pico AVM",
    },
    "particion_trials": {"train": 0.6, "validation": 0.2, "test": 0.2},
    "aumento": {"copias_ruido_gaussiano_caidas": 2, "sigma": 0.01},
    "calibracion": {"ventanas": 1000, "caidas": 600, "adl": 400},
    "entrenamiento": {
        "optimizador": "Adam",
        "tasa_aprendizaje": TASA_APRENDIZAJE,
        "tamano_lote": TAMANO_LOTE,
        "epocas_maximas": EPOCAS_MAXIMAS,
        "parada_desde_epoca": 30,
        "umbral_prediccion": UMBRAL_CAIDA,
    },
}
RUTA_METADATA.write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8"
)

print(f"Modelo guardado: {RUTA_MODELO}")
print(f"Metadatos guardados: {RUTA_METADATA}")
print(f"Datos comprimidos guardados: {RUTA_EVALUACION}")
print("Claves NPZ: x_calibration, y_calibration, x_validation, y_validation, x_test, y_test, x_upfall, y_upfall")